# Imports

In [55]:
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np

#Setup and Loading

- loading csv with pd.read_csv
- using df.shape() and df.columns to find the dimensions and column feature headings
- partin dates into datetime for ease of sorting

In [56]:
df = pd.read_csv("nba_model_ready.csv", encoding="utf-8")
df['date'] = pd.to_datetime(df['date'])
df.head(20)
df.shape

(144874, 24)

In [57]:
# --- STEP A: VERIFY UNIQUE ID (The SQL Fix Check) ---
# If this fails, your SQL truncate/load didn't work.
assert df.duplicated(subset=['player_id', 'game_id']).sum() == 0, "CRITICAL: Duplicates still exist!"

# Sorting the data by date

In [58]:
df_sorted = df.sort_values(by="date", ascending=False)
df_sorted.head(10)

,player_id,game_id,date,season,player_name,opponent_id,min,pts,reb,ast,...,trend_ast,min_l10,fga_per_min_l10,proj_volume,pts_std_l10,pts_mean_l10,cv_l10,opp_avg_pts_allowed_l10,opp_avg_reb_allowed_l10,opp_avg_ast_allowed_l10
128361,1631096,42400407,2025-06-22,2024-25,Chet Holmgren,24,31.17,18,8,0,...,-0.102768,29.493,0.402465,11.869889,6.815831,14.3,0.476632,110.8,42.0,17.8
144111,1642349,42400407,2025-06-22,2024-25,Ajay Mitchell,24,0.32,0,0,0,...,0.220310,4.224,0.370326,1.564255,4.320494,2.0,2.160247,110.8,42.0,17.8
138235,1641717,42400407,2025-06-22,2024-25,Cason Wallace,24,26.03,10,2,0,...,-0.225636,22.349,0.234359,5.237686,2.988868,5.4,0.553494,110.8,42.0,17.8
91194,1629652,42400407,2025-06-22,2024-25,Luguentz Dort,24,35.13,9,7,0,...,-0.023915,30.706,0.197532,6.065431,4.175324,7.9,0.528522,110.8,42.0,17.8
128605,1631097,42400407,2025-06-22,2024-25,Bennedict Mathurin,14,33.23,24,13,3,...,-0.030177,16.244,0.367612,5.971488,9.199034,11.2,0.821342,112.1,42.7,23.8
133505,1631172,42400407,2025-06-22,2024-25,Ousmane Dieng,24,0.32,0,0,0,...,-0.039872,1.581,0.441530,0.698059,2.097618,1.2,1.748015,110.8,42.0,17.8
84242,1629614,42400407,2025-06-22,2024-25,Andrew Nembhard,14,36.41,15,5,6,...,-0.067460,33.077,0.261286,8.642554,4.228212,9.9,0.427092,112.1,42.7,23.8
77325,1629026,42400407,2025-06-22,2024-25,Kenrich Williams,24,4.37,0,1,0,...,0.004952,8.414,0.263139,2.214055,2.867442,2.0,1.433721,110.8,42.0,17.8
54640,1627936,42400407,2025-06-22,2024-25,Alex Caruso,24,32.25,10,3,0,...,0.023938,25.984,0.274543,7.133730,6.789698,9.1,0.746121,110.8,42.0,17.8
41059,1626167,42400407,2025-06-22,2024-25,Myles Turner,14,23.49,6,4,1,...,-0.043871,27.600,0.307583,8.489289,4.880801,11.6,0.420759,112.1,42.7,23.8


# Missing Values and Dtype notation

In [59]:
missing = df.isna().mean().sort_values(ascending=False)
missing[missing > 0]

pts_std_l10     0.011224
pts_mean_l10    0.005764
dtype: float64

In [60]:
df = df.sort_values(by = "date", ascending=True)

new_df = df.dropna(subset = ["pts_mean_l10", "min_l10"])
print(df.shape)

print(new_df.shape)

(144874, 24)
(144039, 24)


# Duplicate Checks


This duplicate check using the duplicated() method with subset = ["player_id", "game_id"] reveales that our data has mismatching duplicated likely due to our transition form pulling data from the nba api to using the kaggle dataset. 

So one api, for one unique player_id, game_id pair, slighly different values for all columns. 


In [61]:

# dup = df.duplicated(subset=["player_id", "game_id"]).sum()
# print("Duplicate player-game rows:", dup)

# # View the duplicate pairs to diagnose the cause
# duplicates = df[df.duplicated(subset=['player_id', 'game_id'], keep=False)]
# print(duplicates.sort_values(['player_id', 'game_id']).head(10))

The differences are negligible. We fix by keeping the precision data (data that is more precise) using df.drop_duplicates(subset=["player_id", "game_id"], keep="first")

In [62]:
# df = df.sort_values('min', ascending=False)
# df_clean = df.drop_duplicates(subset=["player_id", "game_id"], keep = "first")
# df_clean.head(10)
# # print(df_clean.shape)

# HANDLE "DNPs" (Did Not Play) ---
For a prop betting model, we usually only care about games where the player actually stepped on the court.
If min = 0, it's noise.

In [63]:

print(f"Dropping {len(df[df['min'] <= 0])} rows where player played 0 minutes.")
df = df[df['min'] > 0].copy()

Dropping 0 rows where player played 0 minutes.


# --- STEP C: HANDLE MISSING HISTORY (The "First 10 Games" Problem) ---
Columns like 'pts_mean_l10' will be NaN for a player's first 10 games.

We cannot impute these with 0 because that implies they are "bad" players. 
We must drop them to train on high-confidence data.

In [64]:
# --- STEP C: HANDLE MISSING HISTORY (Dynamic & Robust) ---

# 1. Define the patterns that indicate a "Historical/Rolling" feature
#    - '_l10':  Standard rolling average/std (pts_mean_l10, min_l10, cv_l10)
#    - 'trend_': Linear trends (trend_pts, trend_reb)
#    - 'opp_avg_': Opponent defense metrics (opp_avg_pts_allowed_l10)
#    - 'proj_':  Derived projections (proj_volume)
history_patterns = ['_l10', 'trend_', 'opp_avg_', 'proj_']

# 2. Dynamically grab all columns that match these patterns
history_cols = [
    col for col in df.columns 
    if any(pattern in col for pattern in history_patterns)
]

print(f" identified {len(history_cols)} historical feature columns:")
print(history_cols) 
# Should print: ['trend_pts', 'trend_reb', 'trend_ast', 'min_l10', 
#                'fga_per_min_l10', 'proj_volume', 'pts_std_l10', 
#                'pts_mean_l10', 'cv_l10', 'opp_avg_pts_allowed_l10'...]

# 3. Drop rows where ANY of these critical features are NaN
before_drop = len(df)
df = df.dropna(subset=history_cols)
rows_dropped = before_drop - len(df)

print(f"Dropped {rows_dropped} rows (Early season games with insufficient history).")
print(f"Remaining training rows: {len(df)}")

 identified 12 historical feature columns:
['trend_pts', 'trend_reb', 'trend_ast', 'min_l10', 'fga_per_min_l10', 'proj_volume', 'pts_std_l10', 'pts_mean_l10', 'cv_l10', 'opp_avg_pts_allowed_l10', 'opp_avg_reb_allowed_l10', 'opp_avg_ast_allowed_l10']
Dropped 1626 rows (Early season games with insufficient history).
Remaining training rows: 143248


# DATA TYPES OPTIMIZATION

In [69]:
# 1. Dates
df['date'] = pd.to_datetime(df['date'])

# 2. Categoricals (IDs should be strings, not math numbers)
# You don't want the model to think ID 2544 is "greater than" ID 20.
id_cols = ['player_id', 'game_id', 'opponent_id', 'season']
for col in id_cols:
    df[col] = df[col].astype(str)
    
    
df.head(10)

,player_id,game_id,date,season,player_name,opponent_id,min,pts,reb,ast,...,trend_ast,min_l10,fga_per_min_l10,proj_volume,pts_std_l10,pts_mean_l10,cv_l10,opp_avg_pts_allowed_l10,opp_avg_reb_allowed_l10,opp_avg_ast_allowed_l10
20711,203109,41900404,2020-10-06,2020-21,Jae Crowder,5,34.0,8,7,2,...,-0.737916,30.5,0.205882,6.279412,0.000000,12.0,0.000000,113.8,37.2,25.2
60395,1628398,41900404,2020-10-06,2020-21,Kyle Kuzma,18,18.0,9,2,0,...,-0.737916,21.5,0.333333,7.166667,5.656854,15.0,0.377124,109.9,42.6,25.1
684,2738,41900404,2020-10-06,2020-21,Andre Iguodala,5,20.0,3,1,0,...,1.475832,20.5,0.150000,3.075000,3.535534,4.5,0.785674,113.8,37.2,25.2
80513,1629130,41900404,2020-10-06,2020-21,Duncan Robinson,5,32.0,17,1,3,...,-0.737916,30.0,0.218750,6.562500,2.828427,11.0,0.257130,113.8,37.2,25.2
10365,201980,41900404,2020-10-06,2020-21,Danny Green,18,20.0,10,2,1,...,-0.737916,19.0,0.400000,7.600000,0.707107,2.5,0.282843,109.9,42.6,25.1
54291,1627936,41900404,2020-10-06,2020-21,Alex Caruso,18,22.0,7,2,0,...,0.000000,24.5,0.227273,5.568182,1.414214,7.0,0.202031,109.9,42.6,25.1
3,2544,41900404,2020-10-06,2020-21,LeBron James,18,38.0,28,12,8,...,-0.737916,39.0,0.421053,16.421053,5.656854,29.0,0.195064,109.9,42.6,25.1
17121,202710,41900404,2020-10-06,2020-21,Jimmy Butler,5,43.0,22,10,9,...,0.000000,44.0,0.395349,17.395349,10.606602,32.5,0.326357,113.8,37.2,25.2
88343,1629639,41900404,2020-10-06,2020-21,Tyler Herro,5,37.0,21,7,3,...,-0.737916,39.0,0.486486,18.972973,0.000000,17.0,0.000000,113.8,37.2,25.2
1729,200765,41900404,2020-10-06,2020-21,Rajon Rondo,18,28.0,2,7,5,...,-3.689580,26.5,0.250000,6.625000,8.485281,10.0,0.848528,109.9,42.6,25.1


# FINAL LEAKAGE CHECK
Ensure your Features dataframe (X) DOES NOT contain the Targets (y)

In [70]:
target_cols = ['pts', 'reb', 'ast', 'min', 'fga', 'fgm']
feature_cols = [c for c in df.columns if c not in target_cols and c not in id_cols + ['player_name']]

print("\n--- READY FOR TRAINING ---")
print(f"Features (X): {feature_cols}")
print(f"Targets (y): {target_cols}")
print(f"Final Shape: {df.shape}")


--- READY FOR TRAINING ---
Features (X): ['date', 'trend_pts', 'trend_reb', 'trend_ast', 'min_l10', 'fga_per_min_l10', 'proj_volume', 'pts_std_l10', 'pts_mean_l10', 'cv_l10', 'opp_avg_pts_allowed_l10', 'opp_avg_reb_allowed_l10', 'opp_avg_ast_allowed_l10']
Targets (y): ['pts', 'reb', 'ast', 'min', 'fga', 'fgm']
Final Shape: (143248, 24)


In [71]:
df.to_csv("nba_model_ready_clean.csv", index = False)